### Data Reading JSON

In [0]:
df_json=spark.read.format('json').option('inferschema',True).option('Header',True).load('/Volumes/data/default/basav_files/drivers.json')

In [0]:
df_json.display()

###Data Reading

In [0]:
dbutils.fs.ls('/Volumes/data/default/basav_files')

In [0]:
df = spark.read.format('csv').option('inferschema',True).option('header',True).load('/Volumes/data/default/basav_files/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

### SCHEMA -- DDL

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight Double,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df = spark.read.format('csv').schema(my_ddl_schema).option('header',True).load('/Volumes/data/default/basav_files/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

### SELECT

In [0]:
df.printSchema()

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

### ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ident'),col('Item_Weight'),col('Item_Fat_Content')).display()

###FILTER

####Scenario 1

In [0]:
df.filter(col('Item_Fat_Content')== 'Regular').display()

####Scenario 2

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()

####Scenario 3

In [0]:
df.filter((col('outlet_size').isNull()) & (col('Outlet_Location_Type').isin('Tier 2','Tier 1'))).display()

###Column Rename

In [0]:
df.withColumnRenamed('Item_Weight','Item.wt').display()

### With Column

####Sceanrio 1

In [0]:
df = df.withColumn('Flag',lit('Old'))
df.display()

####Scenario 2 

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

#### Scenario 3

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','Regular','Reg'))\
    .withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','Low Fat','LF')).display()

### Type Casting

In [0]:
df= df.withColumn('Item_weight',col('Item_Weight').cast(StringType()))
df.printSchema()
df.display()

### Sort

In [0]:
df.sort(col('Item_weight').asc()).display()

In [0]:
df.sort(['item_weight','item_visibility'],ascending=[0,0]).display()

In [0]:
df.sort(['item_weight','item_visibility'],ascending=[1,0]).display()

###Limit

In [0]:
df.limit(10).display()

###DROP

In [0]:
df.drop('Flag','Item_Weight').display()

###Drop Duplicates
####Scenario 1

In [0]:
df.dropDuplicates().display()

####scenario 2

In [0]:
df.dropDuplicates(subset=['item_type']).display()

In [0]:
df.distinct().display()

###Union

In [0]:
data1=[('1','Basav'),
       ('2','Chetu')]
schema1='id string,name string'
df1=spark.createDataFrame(data1,schema1)


data2=[('3','Basu'),
       ('4','Chetana')]
schema2='id string,name string'
df2=spark.createDataFrame(data2,schema2)


data3=[('Basav','1'),
       ('Chetu','2')]
schema3='name string,id string'
df3=spark.createDataFrame(data3,schema3)

df1.display()
df2.display()
df3.display()

df1.union(df2).display() #### displays union of 2 data frames
df3.union(df2).display() #### Does not map the columns as per types
df3.unionByName(df2).display() #### Used to map column types when they are not in order

###String Functions

####Initcap, upper and lower

In [0]:
df.select(initcap('Item_type')).display()
df.select(upper('item_type')).display()
df.select(lower('item_type')).display()
df.select(concat('item_type','item_identifier')).display()
df.select(concat_ws(' ', 'item_type','item_identifier')).display()
df.select(length('item_type')).display()
df.select(upper('item_type').alias('Upper_item_type')).display()

###DATE

In [0]:
df=df.withColumn('Curr_date',current_date()) #current_date
df.display()
df=df.withColumn('Next_week',date_add(current_date(),7))
df.display() #date_add
df.withColumn('Prev_week',date_sub(current_date(),7)).display() #date_sub
df=df.withColumn('Prev_week',date_add(current_date(),-7)) #works same as date_sub()
df.display()
df.withColumn('Date_diff',datediff('Curr_date','Prev_week')).display()
df.withColumn('Prev_week',date_format('Prev_week','dd-MM-yyyy')).display()

###Handling NULLS

####Dropping NUllS

In [0]:
df.dropna('any').display() #drops entire record if any of the column contains null value
df.dropna('all').display() #drops record only if all the column contains null value
df.dropna(subset=['Outlet_Size']).display() #drops record only if the column 'outslet_size' contains null value
df.dropna(how='any',subset=['Outlet_Size']).display() #same as above


####Filling Nulls

In [0]:
df.fillna('NotAvailable').display() #replaces all null with NotAvailable across all columns
df.fillna('NotAvailable', subset=['Outlet_Size']).display()

###SPLIT and INDEXING

####SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

####Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

###Explode

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### Array Contains

In [0]:
df_exp.display()
df_exp.withColumn('Type_flag',array_contains('Outlet_Type','Type1')).display()

### Group By

#### Scenario 1

In [0]:
df.groupBy('Item_type').agg(sum('Item_MRP')).display()

### Scenario 2

In [0]:
df.groupBy('Item_type').agg(avg('Item_MRP').alias('Avg_MRP')).display()

#### Scenario 3

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

#### Scenario 4

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP'),avg('Item_MRP').alias('Avg_MRP')).display()

### COLLECT LIST

In [0]:
data1=[('1','Basavaraj'),
       ('1','Nidagundi'),
       ('2','Chetana'),
       ('2','Patil')]
schema1='id string,name string'
df1=spark.createDataFrame(data1,schema1)

df1.groupBy('id').agg(collect_list('name').alias('Full name')).display()

## PIVOT

In [0]:
df.groupBy('Item_Type').pivot('Outlet_size').agg(avg('Item_MRP')).display()

### WHEN OTHERWISE

#### Scenario 1

In [0]:
df_WO=df.withColumn('Flag',when(col('Item_type')=='Meat','Non Veg').otherwise('Veg'))
df_WO.display()

#### Scenario 2

In [0]:
df_WO.withColumn('Price_Range',when((col('Flag')=='Veg') & (col('Item_MRP')<100),'Veg Inexpensive')\
    .when((col('Flag')=='Veg') & (col('Item_MRP')>100),'Veg Expensive')\
        .otherwise('Non Veg')).display()

###JOIN

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

df1.display()
df2.display()

#### Inner Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

#### Left Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

#### Right Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

#### ANTI Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'ANTI').display()

###WINDOW FUNCTIONS

#### ROW NUMBER()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('Row Num',row_number().over(Window.orderBy('Item_Identifier'))).display()

### RANK() and DENSE RANK()

In [0]:
df.withColumn('Rank',rank().over(Window.orderBy('Item_identifier'))).display()
df.withColumn('Rank',rank().over(Window.orderBy('Item_identifier')))\
    .withColumn('DenRan',dense_rank().over(Window.orderBy('Item_identifier'))).display()

    #column desc

df.withColumn('Rank',rank().over(Window.orderBy(col('Item_identifier').desc()))).display()
df.withColumn('Rank',rank().over(Window.orderBy(col('Item_identifier').desc())))\
    .withColumn('DenRan',dense_rank().over(Window.orderBy(col('Item_identifier').desc()))).display()


### User Defined Fucntion

In [0]:
def my_func(x):
  return x*x


In [0]:
my_udf=udf(my_func)
df.withColumn('Item_MRP_square',my_udf('Item_MRP')).display()

### Data Writing

####Append

In [0]:
df.write.format('csv').mode('append').save('/Volumes/data/default/basav_files/append.csv')

#### Overwrite

In [0]:
df.write.format('csv').mode('overwrite').save('/Volumes/data/default/basav_files/overwrite.csv')

#### Error

In [0]:
df.write.format('csv')\
  .mode('error')\
  .save('/Volumes/data/default/basav_files/overwrite.csv') # this throws error saying overwrite.csv already exists

In [0]:
df.write.format('csv')\
  .mode('ignore')\
  .save('/Volumes/data/default/basav_files/overrite.csv')

###PARQUET

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .save('/Volumes/data/default/basav_files/data.csv')


###Table

In [0]:
df.write.format('delta')\
.mode('overwrite')\
.saveAsTable('my_table')

df.display()

###SQL

In [0]:
df.createTempView('my_view')

In [0]:
%sql

select * from my_view where Item_fat_content = 'LF'

In [0]:
#store sql result as data frame
my_sql_result = spark.sql("select * from my_view where Item_fat_content = 'LF'")
my_sql_result.display()